Queens Solving Agent

In [ ]:
import pyautogui as pag
import time
import math
from PIL import Image

In [ ]:
def pixelPosition(r, c, d, coords_11, coords_dd):       #row, column, grid dxd, coords (1,1), coords (d,d)
    l_cell = (coords_dd[0] - coords_11[0])/(d - 1)
    return coords_11[0] + round(c*l_cell), coords_11[1] + round(r*l_cell) #returns coords


In [ ]:
# We define the function capable of memorizing the color of each cell of the grid
# Does not take any argument as a input and returns a grid-like array where the content of each cell
# is a number identifying its color

def defineColoredRegions(d):
    colors = [None for _ in range(d**2)]
    for i in range(d):
        for j in range(d):
            x, y = pixelPosition(i,j,d,(x_1, y_1),(x_d, y_d))
            colors[i*d+j] = pag.pixel(x, y)
    return colors   

In [ ]:
d = 9

In [ ]:
time.sleep(1.)
x_1, y_1 = pag.position()
x_1, y_1


In [ ]:
time.sleep(1)
x_d, y_d = pag.position()
x_d, y_d

In [ ]:
pag.moveTo(10,10) #così mi levo da sopra le celle perchè il colore è diverso se no
cells_colors = defineColoredRegions(d)

In [ ]:
def getColoredRegionCells(g, c):
    cells = []
    for i in range(d):
        for j in range(d):
            if cells_colors[i*d+j] == c:
                cells.append(g[i*d + j])
    return cells

In [ ]:
for x in range(d):
    print(cells_colors[x*d:x*d+d])

[(144, 51, 103), (174, 60, 122), (185, 64, 130), (185, 64, 130), (185, 64, 130), (185, 64, 130), (185, 64, 130), (185, 64, 130), (185, 64, 130)]
[(64, 43, 28), (72, 77, 86), (108, 114, 127), (132, 77, 50), (132, 77, 50), (132, 77, 50), (132, 77, 50), (132, 77, 50), (185, 64, 130)]
[(184, 113, 57), (93, 98, 109), (65, 47, 115), (82, 59, 146), (112, 79, 199), (112, 79, 199), (132, 77, 50), (20, 149, 112), (185, 64, 130)]
[(196, 119, 60), (52, 107, 199), (47, 36, 85), (65, 47, 115), (46, 44, 25), (20, 149, 112), (20, 149, 112), (20, 149, 112), (185, 64, 130)]
[(196, 119, 60), (52, 107, 199), (112, 79, 199), (81, 75, 36), (111, 101, 44), (185, 64, 130), (185, 64, 130), (20, 149, 112), (185, 64, 130)]
[(196, 119, 60), (52, 107, 199), (112, 79, 199), (112, 79, 199), (185, 64, 130), (185, 64, 130), (185, 64, 130), (20, 149, 112), (185, 64, 130)]
[(196, 119, 60), (52, 107, 199), (52, 107, 199), (150, 23, 60), (185, 64, 130), (185, 64, 130), (185, 64, 130), (185, 64, 130), (185, 64, 130)]
[(196

In [ ]:
colors = []
for i in range(d):
    for j in range(d):
        if cells_colors[i*d+j] not in colors:
            colors.append(cells_colors[i*d+j])
        cells_colors[i*d+j] = colors.index(cells_colors[i*d+j])

for x in range(d):
    print(cells_colors[x*d:x*d+d])

[0, 1, 2, 2, 2, 2, 2, 2, 2]
[3, 4, 5, 6, 6, 6, 6, 6, 2]
[7, 8, 9, 10, 11, 11, 6, 12, 2]
[13, 14, 15, 9, 16, 12, 12, 12, 2]
[13, 14, 11, 17, 18, 2, 2, 12, 2]
[13, 14, 11, 11, 2, 2, 2, 12, 2]
[13, 14, 14, 19, 2, 2, 2, 2, 2]
[13, 14, 19, 19, 19, 19, 2, 2, 2]
[13, 13, 13, 13, 13, 13, 2, 2, 2]


In [ ]:
def getCol(g, j):
    result = []
    for x in range(d**2):
        if x%d == j:
            result.append(g[x])
    return result
        
def getRow(g, i):
    result = []
    for x in range(d**2):
        if x//d == i:
            result.append(g[x])
    return result
        

def checkRow(g, i):       
    return 1 not in getRow(g, i) # Tells me whether that row is available

def checkColumn(g, j):
    return 1 not in getCol(g, j)

def checkColoredRegion(g, c):
    return 1 not in getColoredRegionCells(g, c)

def checkDiagonalsEmpty(grid, idx):
    """
    Controlla se le 4 celle diagonali attorno alla cella idx sono tutte 0.
    
    grid: lista 1D flattenata di dimensione d*d
    idx: indice della cella da controllare (0 ... d*d-1)
    d: dimensione della griglia
    """
    i = idx // d  # riga
    j = idx % d   # colonna

    # direzioni diagonali: (di, dj)
    diagonals = [(-1,-1), (-1,1), (1,-1), (1,1)]

    for di, dj in diagonals:
        ni, nj = i + di, j + dj
        if 0 <= ni < d and 0 <= nj < d:
            nidx = ni * d + nj
            if grid[nidx] != 0:
                return False
    return True

In [ ]:
def solveQueens(g, row):           #Solve grid starting from row
   
    for col in range(d):
        idx = row*d+col

        if checkColumn(g, col) and checkColoredRegion(g, cells_colors[idx]) and checkDiagonalsEmpty(g, idx):
            g[idx] = 1

            if row == d-1:
                return sum(g) == d
    
            if solveQueens(g, row + 1):
                return True
            
            g[idx] = 0
            
    return False

In [ ]:
grid = [0 for _ in range(d**2)]
for x in range(d):
    print(grid[x*d:x*d+d])

[0, 0, 0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0, 0, 0]


In [ ]:
solveQueens(grid, 0)

True

In [ ]:
for x in range(d):
    print(grid[x*d:x*d+d])

[1, 0, 0, 0, 0, 0, 0, 0, 0]
[0, 0, 1, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 1, 0, 0]
[0, 1, 0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 1, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0, 1, 0]
[0, 0, 0, 1, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0, 0, 1]
[0, 0, 0, 0, 0, 1, 0, 0, 0]


In [ ]:
# compila griglia
for i in range(d):
    for j in range(d):
        if grid[i*d + j] == 1:
            x, y = pixelPosition(i,j,d,(x_1, y_1),(x_d, y_d))
            pag.moveTo(x, y)
            pag.rightClick()
        else:
            pass
    